In [5]:
import urllib
from tqdm import tqdm
import requests
import pandas as pd
import re
import json

## This notebook adds which politician it is, and what their party is


In [7]:
with urllib.request.urlopen(f"https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/voting-data/df_votes_all_periods.csv") as response:
    df = pd.read_csv(response)

print(df["aktørid"].nunique())
df.head(2)

710


,vote_id,vote_typeid,afstemning_id,aktørid,vote_opdateringsdato
0,1481475,3,5973,1886,2021-01-28T21:27:39.627
1,1481476,1,5973,1039,2018-02-16T10:35:23


In [ ]:
# with urllib.request.urlopen(f"https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/party_colors.json") as response:
#     party_colors = json.load(response)

In [ ]:
# party_colors
# parties = []
# for party in party_colors:
#     parties.append()

ValueError: too many values to unpack (expected 2)

In [11]:
request_session = requests.Session()
def get_actors_for_votes(aktørid,  session = request_session):
    base_url = "https://oda.ft.dk/api/"
    item = "Aktør"
    url = f"{base_url}{item}({aktørid})"
    response = session.get(url)
    # print(f"Sent call to URL: {response.url}")
    if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
        print(f"HTTP error for {aktørid}: ", response.status_code)
        print("Response text:", response.text)
        return None
    else:
        try:
            data = response.json()
        except ValueError:
            print("Error: Response is not valid JSON")
            print("Response text:", response.text)
            return None
        
    politician_name = data.get("navn")
    try:
        politician_standard_party = re.search('<party>([^<]+)</party>',  data.get("biografi")).group(1)
    except:
        politician_standard_party = "Not able to assign"
        print(f"Not able to assign party for {politician_name} with id {aktørid}")
        
    return aktørid, politician_name, politician_standard_party

In [ ]:
unique_actors = df['aktørid'].unique()

all_actors = []
for aktørid in tqdm(unique_actors):
    data = get_actors_for_votes(aktørid)
    all_actors.append(data)
actor_df = pd.DataFrame(all_actors)
actor_df.rename(columns = {0: "aktørid", 1 : "politician", 2: "party"}, inplace = True)

actor_df.head()

  0%|          | 0/710 [00:00<?, ?it/s]

  5%|▌         | 39/710 [00:00<00:10, 66.57it/s]

Not able to assign party for Jytte Andersen with id 8319


 12%|█▏        | 83/710 [00:01<00:09, 66.21it/s]

Not able to assign party for Aage Frandsen with id 1623
Not able to assign party for Carsten Hansen with id 5593
Not able to assign party for Torben Hansen with id 7633


 24%|██▍       | 170/710 [00:02<00:10, 49.73it/s]

Not able to assign party for Frode Sørensen, Hjørring with id 3042


 32%|███▏      | 230/710 [00:03<00:08, 57.59it/s]

Not able to assign party for Mogens Jensen, Brøndby with id 5905


100%|██████████| 710/710 [00:11<00:00, 61.57it/s]


In [16]:
actor_df.to_csv("./actor-data/actor_df.csv", index= False)

In [ ]:
request_session = requests.Session()
def get_actor_party(aktør_id,  session = request_session):
    base_url = "https://oda.ft.dk/api/"
    item = "AktørAktør"
    url = f"{base_url}{item}({aktør_id})"
    response = session.get(url)
    # print(f"Sent call to URL: {response.url}")
    if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
        print(f"HTTP error for {aktør_id}: ", response.status_code)
        print("Response text:", response.text)
        return None
    else:
        try:
            data = response.json()
        except ValueError:
            print("Error: Response is not valid JSON")
            print("Response text:", response.text)
            return None
        
    return data

get_actor_party(unique_actors[100])

HTTP error for 62:  404
Response text: 


In [ ]:
#AktørAktørRolle = 15 -> personen er medlem af partiet
#fraaktørid = 158
# Giver medlem (rolle15) af: 1259=SF, 1088=SF, 1315=SF, 1165=SF, 6=Folketinget.
# Partierne har typeid=4
# Vi skal altså filtere på typeid = 4

# tilaktørid = aktør_id
# For hvert id skal vi finde alle AktørAktør relationer med rolleid = 15
# Så skal vi tage startdato og slutdato
# Så skal vi på fraaktørid finde navnet på den aktør, som så vil være et parti
# Så skal vi lave en eller anden smart gruppering så det bliver joinet på dataen

